## Conexão dos Notebooks


<a href="https://colab.research.google.com/github/Yuoneeh/Projeto_Redes_EstDados/blob/main/notebooks/isaque/Tarefa3_Arvore_inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

# Detecta se está no Google Colab ou em ambiente local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROJETO = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    RAIZ_PROJETO = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]

# Configura o caminho absoluto de PROC
PROC = RAIZ_PROJETO / "data" / "processed"

## Arvore Inicial

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

SEED = 42
CLASSES = ["OK", "RISCO", "FALHA"]

# 1. Carregar o dataset rotulado da Tarefa 2 (mesmo corte temporal)
df = pd.read_csv(PROC / "dataset_final_treino.csv")

# Colunas oficiais permitidas na árvore (sem país, IP, rota ou RTT absoluto)
FEATURES = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pacotes",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

# Tratamento operacional de NaNs apenas para os nós de divisão da árvore
# (Em timeout, timeout_atual=1 e perda_pacotes=100 já estão preenchidos)
X = df[FEATURES].fillna({"z_robusto": 0.0, "aumento_pct": 0.0, "jitter_relativo": 1.0})
y = df["status"]

treino_mask = df["bloco"] == "treino"
val_mask = df["bloco"] == "validacao"

X_treino, y_treino = X[treino_mask], y[treino_mask]
X_val, y_val = X[val_mask], y[val_mask]

print("Colunas usadas na árvore oficial:", FEATURES)
print(f"N treino: {len(X_treino)} | N validação: {len(X_val)}")

# ==============================================================================
# ESTRUTURA DE DADOS II: IMPLEMENTAÇÃO MANUAL DA ÁRVORE DE DECISÃO (CART)
# ==============================================================================
class NoArvore:
    def __init__(self, feature=None, limiar=None, esquerda=None, direita=None, *, valor=None, gini=0.0, n_amostras=0):
        self.feature = feature          # Índice da coluna usada na pergunta (nó interno)
        self.limiar = limiar            # Valor de corte (x <= limiar)
        self.esquerda = esquerda        # Ponteiro para subárvore esquerda (Sim)
        self.direita = direita          # Ponteiro para subárvore direita (Não)
        self.valor = valor              # Classe predita se for nó folha ("OK", "RISCO", "FALHA")
        self.gini = gini
        self.n_amostras = n_amostras

    def eh_folha(self):
        return self.valor is not None


class ArvoreDecisaoManualCART:
    def __init__(self, max_depth=3, min_samples_leaf=50):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.raiz = None
        self.nomes_features = None

    def _calcular_gini(self, y_arr):
        n = len(y_arr)
        if n == 0:
            return 0.0
        _, contagens = np.unique(y_arr, return_counts=True)
        proporcoes = contagens / n
        return 1.0 - np.sum(proporcoes ** 2)

    def _classe_majoritaria(self, y_arr):
        valores, contagens = np.unique(y_arr, return_counts=True)
        return valores[np.argmax(contagens)]

    def _melhor_divisao(self, X_arr, y_arr):
        n_amostras, n_features = X_arr.shape
        melhor_ganho = -1.0
        melhor_idx, melhor_limiar = None, None
        gini_pai = self._calcular_gini(y_arr)

        for feat_idx in range(n_features):
            coluna = X_arr[:, feat_idx]
            limiares = np.unique( np.percentile(coluna, np.linspace(5, 95, 19)) )
            for limiar in limiares:
                esq_mask = coluna <= limiar
                dir_mask = ~esq_mask
                n_esq, n_dir = np.sum(esq_mask), np.sum(dir_mask)

                if n_esq < self.min_samples_leaf or n_dir < self.min_samples_leaf:
                    continue

                gini_filhos = (n_esq / n_amostras) * self._calcular_gini(y_arr[esq_mask]) + \
                              (n_dir / n_amostras) * self._calcular_gini(y_arr[dir_mask])
                ganho = gini_pai - gini_filhos

                if ganho > melhor_ganho:
                    melhor_ganho = ganho
                    melhor_idx = feat_idx
                    melhor_limiar = limiar

        return melhor_idx, melhor_limiar, gini_pai

    def _construir_recursivo(self, X_arr, y_arr, profundidade=0):
        n_amostras = len(y_arr)
        n_classes = len(np.unique(y_arr))
        gini_atual = self._calcular_gini(y_arr)

        # Critérios de parada (poda por profundidade, folha mínima ou nó puro)
        if (profundidade >= self.max_depth or n_classes == 1 or n_amostras < 2 * self.min_samples_leaf):
            return NoArvore(valor=self._classe_majoritaria(y_arr), gini=gini_atual, n_amostras=n_amostras)

        feat_idx, limiar, _ = self._melhor_divisao(X_arr, y_arr)
        if feat_idx is None:
            return NoArvore(valor=self._classe_majoritaria(y_arr), gini=gini_atual, n_amostras=n_amostras)

        esq_mask = X_arr[:, feat_idx] <= limiar
        sub_esq = self._construir_recursivo(X_arr[esq_mask], y_arr[esq_mask], profundidade + 1)
        sub_dir = self._construir_recursivo(X_arr[~esq_mask], y_arr[~esq_mask], profundidade + 1)

        return NoArvore(feature=feat_idx, limiar=limiar, esquerda=sub_esq, direita=sub_dir,
                        gini=gini_atual, n_amostras=n_amostras)

    def fit(self, X_df, y_series):
        self.nomes_features = list(X_df.columns)
        self.raiz = self._construir_recursivo(X_df.values, y_series.values, 0)
        return self

    def _prever_linha(self, x, no):
        if no.eh_folha():
            return no.valor
        if x[no.feature] <= no.limiar:
            return self._prever_linha(x, no.esquerda)
        return self._prever_linha(x, no.direita)

    def predict(self, X_df):
        return np.array([self._prever_linha(x, self.raiz) for x in X_df.values])

    def imprimir_regras(self, no=None, prefixo=""):
        if no is None:
            no = self.raiz
        if no.eh_folha():
            print(f"{prefixo}--> Classe: {no.valor} (n={no.n_amostras}, gini={no.gini:.3f})")
        else:
            nome = self.nomes_features[no.feature]
            print(f"{prefixo}Se {nome} <= {no.limiar:.2f}:")
            self.imprimir_regras(no.esquerda, prefixo + "  | ")
            print(f"{prefixo}Senão ({nome} > {no.limiar:.2f}):")
            self.imprimir_regras(no.direita, prefixo + "  | ")

# Treinar a Árvore Manual e exibir regras
arvore_manual = ArvoreDecisaoManualCART(max_depth=3, min_samples_leaf=50)
arvore_manual.fit(X_treino, y_treino)
print("\n--- Regras da Árvore Manual (Estrutura de Dados II) ---")
arvore_manual.imprimir_regras()

# ==============================================================================
# 2. ÁRVORE OFICIAL (SCIKIT-LEARN CART) E LEITURA DO ERRO NA VALIDAÇÃO (TAREFA 3)
# ==============================================================================
arvore_t3 = DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=50, random_state=SEED)
arvore_t3.fit(X_treino, y_treino)  # Fit SOMENTE no treino!

pred_val = arvore_t3.predict(X_val)
f1_macro_t3 = f1_score(y_val, pred_val, average="macro", labels=CLASSES, zero_division=0)
rec_falha_t3 = recall_score(y_val, pred_val, labels=["FALHA"], average="macro", zero_division=0)

print(f"\nProfundidade real: {arvore_t3.get_depth()} | Folhas: {arvore_t3.get_n_leaves()}")
print(f"F1 Macro (Validação): {f1_macro_t3:.4f} | Recall FALHA: {rec_falha_t3:.4f}")
print("\nMatriz 3x3 (OK, RISCO, FALHA) em contagem:")
print(pd.DataFrame(confusion_matrix(y_val, pred_val, labels=CLASSES),
                   index=[f"Real_{c}" for c in CLASSES], columns=[f"Pred_{c}" for c in CLASSES]))
print("\nRelatório por classe:")
print(classification_report(y_val, pred_val, labels=CLASSES, zero_division=0))
print("\nRegras lidas da árvore:")
print(export_text(arvore_t3, feature_names=FEATURES))

# ==============================================================================
# 4. ÁRVORE DE CONTRASTE (COM RTT ABSOLUTO — PROVA DO ATALHO DA DISTÂNCIA)
# ==============================================================================
X_contraste = df.loc[treino_mask, FEATURES + ["latencia", "mediana"]].fillna(0)
arvore_contraste = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=SEED)
arvore_contraste.fit(X_contraste, y_treino)
print("--- Árvore de Contraste (Descartada da entrega final) ---")
print(export_text(arvore_contraste, feature_names=FEATURES + ["latencia", "mediana"]))